# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), stratified 5-fold CV on training rows, choice by mean CV ROC AUC, holdout scored once.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

sha = lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
df = pd.read_csv('data.csv'); hold = pd.read_csv('holdout.csv')
assert len(df)==7000 and sha('data.csv')=="25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4"
assert sha('holdout.csv')=="083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf"
hold_idx = hold['row'].to_numpy()   # 0-based positions in data.csv
assert len(set(hold_idx))==1400 and hold_idx.min()>=0 and hold_idx.max()<len(df)
is_hold = np.zeros(len(df),bool); is_hold[hold_idx]=True
train_idx = np.where(~is_hold)[0]
print(len(train_idx), len(hold_idx), df.claim_within_year.mean())

## Features
All of `channel`, `device_age_years`, `price_eur`, `daily_hours` are recorded at sale/registration, i.e. known at prediction time (see `columns.csv`). `device_serial` is excluded (identifier, per protocol). Note: `daily_hours` has values above 24 (max 34.9), which is implausible for hours per day; kept as reported (a registration-time self-report).

In [ ]:
target='claim_within_year'
cat=['channel']; num=['device_age_years','price_eur','daily_hours']
features=cat+num
X=df[features]; y=df[target]
Xtr, ytr = X.iloc[train_idx], y.iloc[train_idx]
Xho, yho = X.iloc[hold_idx], y.iloc[hold_idx]
print(df[features].describe(include='all'))

## Candidates (all named before any scoring)
Preprocessing lives inside each Pipeline, so it is fitted within CV folds.

In [ ]:
def pre(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),
                              ('n',StandardScaler() if scale else 'passthrough',num)])
candidates = {
 'baseline_prior': Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':  Pipeline([('p',pre(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':    Pipeline([('p',pre(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'rf_depth6':    Pipeline([('p',pre(False)),('m',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hgb_small':    Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_folds = [[[int(train_idx[i]) for i in tr], [int(train_idx[i]) for i in va]] for tr,va in cv.split(Xtr,ytr)]  # [train_rows, val_rows] per fold
cv_scores={}
for name,est in candidates.items():
    s = cross_val_score(est, Xtr, ytr, cv=cv, scoring='roc_auc')
    cv_scores[name]=[float(v) for v in s]
    print(f"{name:16s} mean={s.mean():.4f} sd={s.std():.4f}")
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:',chosen)

## Holdout: scored once, chosen candidate refitted on all training rows

In [ ]:
final = candidates[chosen].fit(Xtr, ytr)
p = final.predict_proba(Xho)[:,1]
holdout_auc = float(roc_auc_score(yho, p))
print(f"Holdout ROC AUC ({chosen}): {holdout_auc:.4f}")

In [ ]:
evidence = {
 'features': features,
 'candidates': list(candidates),
 'cv_folds': cv_folds,
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': [int(i) for i in hold_idx],
 'holdout_scores': {chosen: holdout_auc}  # roc_auc, chosen candidate only,
}
json.dump(evidence, open('evidence.json','w'))